In [0]:
df_contas = spark.table(
    "workspace.desafio_sicredi.bronze_contas"
)

display(df_contas)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "workspace.desafio_sicredi"

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "workspace.desafio_sicredi"

df_contas = spark.table(f"{BASE}.bronze_contas")

janela = Window.partitionBy("id_conta").orderBy(
    F.col("arquivo_origem").desc()
)

df_contas_dedup = (
    df_contas
    .withColumn("rn", F.row_number().over(janela))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(
    df_contas_dedup.select(
        "id_conta",
        "id_cliente",
        "tipo_conta",
        "status_conta"
    )
)

id_conta,id_cliente,tipo_conta,status_conta
101,1,CORRENTE,ATIVA
102,2,POUPANCA,ATIVA
103,3,CORRENTE,BLOQUEADA
104,999,CORRENTE,ATIVA
105,1,POUPANCA,DESCONHECIDO


In [0]:
total = df_contas_dedup.count()

distintos = (
    df_contas_dedup
    .select("id_conta")
    .distinct()
    .count()
)

assert total == distintos, "Ainda existem contas duplicadas!"

print(f"PASSOU — {total} contas únicas")

PASSOU — 5 contas únicas


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Converter os tipos de dados
df_contas_tipadas = (
    df_contas
    .withColumn(
        "id_conta",
        F.expr("try_cast(id_conta AS INT)")
    )
    .withColumn(
        "id_cliente",
        F.expr("try_cast(id_cliente AS INT)")
    )
    .withColumn(
        "data_abertura",
        F.expr("try_cast(data_abertura AS DATE)")
    )
)

# 2. Remover duplicidades pelo identificador da conta
janela = Window.partitionBy("id_conta").orderBy(
    F.col("arquivo_origem").desc()
)

df_contas_tipadas = (
    df_contas_tipadas
    .withColumn("rn", F.row_number().over(janela))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 3. Exibir resultado
display(df_contas_tipadas)

id_conta,id_cliente,tipo_conta,status_conta,data_abertura,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha
101,1,CORRENTE,ATIVA,2024-01-10,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,5adc75a014101edf6283f9ddd25388d92679363a61706fa7a3f237083031e97d
102,2,POUPANCA,ATIVA,2024-02-15,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,4d44edc3cdd1ee11a6645f48d507eb531a53b5f7b59b8243ba7373e5444e4c07
103,3,CORRENTE,BLOQUEADA,2024-03-20,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,fe60b05f9da19e6745c0e76bd97f9be656041b2ad28e8a1344939e19f5c26971
104,999,CORRENTE,ATIVA,2024-04-10,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,c94552649cf8992bb6813f51bcaa83a7e7b23fbe961280cbb53c5e9ab7d86395
105,1,POUPANCA,DESCONHECIDO,2024-05-01,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,ca14db58737c3ec08fdb62ce590e340c52a2c5689290dd57fc7b5eb376ebe436


In [0]:
from pyspark.sql import functions as F

# Ler clientes válidos da Silver
df_clientes_silver = spark.table(
    "workspace.desafio_sicredi.silver_clientes"
)

# Selecionar os IDs dos clientes válidos
df_clientes_ids = (
    df_clientes_silver
    .select("id_cliente")
    .distinct()
    .withColumn("cliente_existe", F.lit(True))
)

# Validar integridade referencial
df_contas_integridade = (
    df_contas_tipadas
    .join(
        df_clientes_ids,
        on="id_cliente",
        how="left"
    )
)

# Visualizar o resultado
display(df_contas_integridade)

id_cliente,id_conta,tipo_conta,status_conta,data_abertura,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha,cliente_existe
1,101,CORRENTE,ATIVA,2024-01-10,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-1212860819587999444-03182888-1d66-4766-9251-5cb35f673774-141-1-c000.csv,null,2026-10-15T17:09:27.554Z,5adc75a014101edf6283f9ddd25388d92679363a61706fa7a3f237083031e97d,true
2,102,POUPANCA,ATIVA,2024-02-15,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-1212860819587999444-03182888-1d66-4766-9251-5cb35f673774-141-1-c000.csv,null,2026-10-15T17:09:27.554Z,4d44edc3cdd1ee11a6645f48d507eb531a53b5f7b59b8243ba7373e5444e4c07,null
3,103,CORRENTE,BLOQUEADA,2024-03-20,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-1212860819587999444-03182888-1d66-4766-9251-5cb35f673774-141-1-c000.csv,null,2026-10-15T17:09:27.554Z,fe60b05f9da19e6745c0e76bd97f9be656041b2ad28e8a1344939e19f5c26971,true
999,104,CORRENTE,ATIVA,2024-04-10,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-1212860819587999444-03182888-1d66-4766-9251-5cb35f673774-141-1-c000.csv,null,2026-10-15T17:09:27.554Z,c94552649cf8992bb6813f51bcaa83a7e7b23fbe961280cbb53c5e9ab7d86395,null
1,105,POUPANCA,DESCONHECIDO,2024-05-01,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-1212860819587999444-03182888-1d66-4766-9251-5cb35f673774-141-1-c000.csv,null,2026-10-15T17:09:27.554Z,ca14db58737c3ec08fdb62ce590e340c52a2c5689290dd57fc7b5eb376ebe436,true
1,101,CORRENTE,ATIVA,2024-01-10,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,5adc75a014101edf6283f9ddd25388d92679363a61706fa7a3f237083031e97d,true
2,102,POUPANCA,ATIVA,2024-02-15,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,4d44edc3cdd1ee11a6645f48d507eb531a53b5f7b59b8243ba7373e5444e4c07,null
3,103,CORRENTE,BLOQUEADA,2024-03-20,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,fe60b05f9da19e6745c0e76bd97f9be656041b2ad28e8a1344939e19f5c26971,true
999,104,CORRENTE,ATIVA,2024-04-10,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,c94552649cf8992bb6813f51bcaa83a7e7b23fbe961280cbb53c5e9ab7d86395,null
1,105,POUPANCA,DESCONHECIDO,2024-05-01,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-3565500965905918807-57f42d91-86be-4d47-8944-57175a5fe7dd-161-1-c000.csv,null,2026-10-15T13:48:05.138Z,ca14db58737c3ec08fdb62ce590e340c52a2c5689290dd57fc7b5eb376ebe436,true


In [0]:
from pyspark.sql import functions as F

# ETAPA 1 - Ler contas da Bronze
df_contas = spark.table(
    "workspace.desafio_sicredi.bronze_contas"
)

# ETAPA 2 - Converter os tipos
df_contas_tipadas = (
    df_contas
    .withColumn(
        "id_conta",
        F.expr("try_cast(id_conta AS INT)")
    )
    .withColumn(
        "id_cliente",
        F.expr("try_cast(id_cliente AS INT)")
    )
    .withColumn(
        "data_abertura",
        F.expr("try_cast(data_abertura AS DATE)")
    )
)

# ETAPA 3 - Ler clientes válidos da Silver
df_clientes_silver = spark.table(
    "workspace.desafio_sicredi.silver_clientes"
)

# ETAPA 4 - Selecionar clientes válidos
df_clientes_ids = (
    df_clientes_silver
    .select("id_cliente")
    .distinct()
    .withColumn(
        "cliente_existe",
        F.lit(True)
    )
)

# ETAPA 5 - Validar integridade referencial
df_contas_integridade = (
    df_contas_tipadas
    .join(
        df_clientes_ids,
        on="id_cliente",
        how="left"
    )
)

# ETAPA 6 - Exibir resultado
display(
    df_contas_integridade.select(
        "id_conta",
        "id_cliente",
        "tipo_conta",
        "status_conta",
        "cliente_existe"
    )
)

id_conta,id_cliente,tipo_conta,status_conta,cliente_existe
101,1,CORRENTE,ATIVA,true
102,2,POUPANCA,ATIVA,null
103,3,CORRENTE,BLOQUEADA,true
104,999,CORRENTE,ATIVA,null
105,1,POUPANCA,DESCONHECIDO,true
101,1,CORRENTE,ATIVA,true
102,2,POUPANCA,ATIVA,null
103,3,CORRENTE,BLOQUEADA,true
104,999,CORRENTE,ATIVA,null
105,1,POUPANCA,DESCONHECIDO,true


In [0]:
##regras_qualidade

df_contas_qualidade = (
    df_contas_integridade
    .withColumn(
        "motivo_erro",
        F.when(
            F.col("id_conta").isNull(),
            "ID_CONTA_INVALIDO"
        )
        .when(
            F.col("id_cliente").isNull(),
            "ID_CLIENTE_INVALIDO"
        )
        .when(
            F.col("cliente_existe").isNull(),
            "CLIENTE_INEXISTENTE_OU_INVALIDO"
        )
        .when(
            ~F.col("status_conta").isin(
                "ATIVA",
                "INATIVA",
                "BLOQUEADA",
                "ENCERRADA"
            ),
            "STATUS_CONTA_INVALIDO"
        )
        .when(
            F.col("data_abertura").isNull(),
            "DATA_ABERTURA_INVALIDA"
        )
    )
)

In [0]:
df_contas_validas = (
    df_contas_qualidade
    .filter(F.col("motivo_erro").isNull())
    .drop("cliente_existe", "motivo_erro")
)

df_contas_quarentena = (
    df_contas_qualidade
    .filter(F.col("motivo_erro").isNotNull())
    .drop("cliente_existe")
)

print("Contas válidas:", df_contas_validas.count())
print("Contas em quarentena:", df_contas_quarentena.count())

Contas válidas: 2
Contas em quarentena: 3


In [0]:
(
    df_contas_validas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.desafio_sicredi.silver_contas"
    )
)

(
    df_contas_quarentena.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.desafio_sicredi.quarentena_contas"
    )
)

In [0]:
##ler_cartoes

df_cartoes = spark.table(
    "workspace.desafio_sicredi.bronze_cartoes"
)

df_cartoes_tipados = (
    df_cartoes
    .withColumn(
        "id_cartao",
        F.expr("try_cast(id_cartao AS INT)")
    )
    .withColumn(
        "id_conta",
        F.expr("try_cast(id_conta AS INT)")
    )
    .withColumn(
        "data_emissao",
        F.expr("try_cast(data_emissao AS DATE)")
    )
    .withColumn(
        "data_atualizacao",
        F.expr("try_cast(data_atualizacao AS TIMESTAMP)")
    )
)

display(df_cartoes_tipados)

id_cartao,id_conta,bandeira,tipo_cartao,status_cartao,data_emissao,data_atualizacao,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha
1001,101,VISA,CREDITO,ATIVO,2024-01-15,2026-10-13T08:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,3f03a35648a32ff16f189b066bc29ada92ceba7bc6c9ba392043ac99a4f7a6a2
1001,101,VISA,CREDITO,CANCELADO,2024-01-15,2026-10-15T09:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,b80bc8911b1b0d74f4e69de469b0d92c5ea8fd5f4a8026b58a619f1f6d4bb379
1002,102,MASTERCARD,DEBITO,ATIVO,2024-02-20,2026-10-13T10:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,7f9c291d3ddaac544f8ff0c56f1a4588a2231b4f6776dbcdddbaed1fb876e5d1
1003,103,ELO,CREDITO,ATIVO,2024-03-25,2026-10-14T11:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,317d4b9ad8fb5864dcb58a1a764a665f1c4c32d970e02569e681bda300df3e04
1004,999,VISA,CREDITO,ATIVO,2024-04-10,2026-10-15T12:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,e7065d134daedf19b098e7796615fb30d7cd60a32525762b8344fc83d93faf07
1005,101,MASTERCARD,CREDITO,ATIVO,2024-05-01,2026-10-13T08:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,3b8c46fc447c49ad20771d37e26626ba8467082b55147b3bdcd41159a68ce787


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Criar janela para identificar a versão mais recente
janela_cartoes = (
    Window
    .partitionBy("id_cartao")
    .orderBy(
        F.col("data_atualizacao").desc_nulls_last(),
        F.col("timestamp_ingestao").desc_nulls_last(),
        F.col("hash_linha").desc()
    )
)

# Deduplicar os cartões
df_cartoes_dedup = (
    df_cartoes_tipados
    .withColumn(
        "rn",
        F.row_number().over(janela_cartoes)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Exibir resultado
display(df_cartoes_dedup)

id_cartao,id_conta,bandeira,tipo_cartao,status_cartao,data_emissao,data_atualizacao,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha
1001,101,VISA,CREDITO,CANCELADO,2024-01-15,2026-10-15T09:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,b80bc8911b1b0d74f4e69de469b0d92c5ea8fd5f4a8026b58a619f1f6d4bb379
1002,102,MASTERCARD,DEBITO,ATIVO,2024-02-20,2026-10-13T10:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,7f9c291d3ddaac544f8ff0c56f1a4588a2231b4f6776dbcdddbaed1fb876e5d1
1003,103,ELO,CREDITO,ATIVO,2024-03-25,2026-10-14T11:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,317d4b9ad8fb5864dcb58a1a764a665f1c4c32d970e02569e681bda300df3e04
1004,999,VISA,CREDITO,ATIVO,2024-04-10,2026-10-15T12:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,e7065d134daedf19b098e7796615fb30d7cd60a32525762b8344fc83d93faf07
1005,101,MASTERCARD,CREDITO,ATIVO,2024-05-01,2026-10-13T08:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1247668048967399409-e51c5724-92d4-47f1-a566-940a89012665-162-1-c000.csv,null,2026-10-15T13:48:12.159Z,3b8c46fc447c49ad20771d37e26626ba8467082b55147b3bdcd41159a68ce787


In [0]:
df_contas_silver = spark.table(
    "workspace.desafio_sicredi.silver_contas"
)

df_cartoes_integridade = (
    df_cartoes_dedup
    .join(
        df_contas_silver
        .select("id_conta")
        .withColumn("conta_existe", F.lit(True)),
        on="id_conta",
        how="left"
    )
)

In [0]:
df_cartoes_qualidade = (
    df_cartoes_integridade
    .withColumn(
        "motivo_erro",
        F.when(
            F.col("id_cartao").isNull(),
            "ID_CARTAO_INVALIDO"
        )
        .when(
            F.col("id_conta").isNull(),
            "ID_CONTA_INVALIDO"
        )
        .when(
            F.col("conta_existe").isNull(),
            "CONTA_INEXISTENTE_OU_INVALIDA"
        )
        .when(
            ~F.col("status_cartao").isin(
                "ATIVO",
                "BLOQUEADO",
                "CANCELADO"
            ),
            "STATUS_CARTAO_INVALIDO"
        )
        .when(
            F.col("data_emissao").isNull(),
            "DATA_EMISSAO_INVALIDA"
        )
        .when(
            F.col("data_atualizacao").isNull(),
            "DATA_ATUALIZACAO_INVALIDA"
        )
    )
)

In [0]:
df_cartoes_validos = (
    df_cartoes_qualidade
    .filter(F.col("motivo_erro").isNull())
    .drop("motivo_erro", "conta_existe")
)

df_cartoes_quarentena = (
    df_cartoes_qualidade
    .filter(F.col("motivo_erro").isNotNull())
    .drop("conta_existe")
)

print("Cartões válidos:", df_cartoes_validos.count())
print("Cartões em quarentena:", df_cartoes_quarentena.count())

Cartões válidos: 3
Cartões em quarentena: 2


In [0]:
(
    df_cartoes_validos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.desafio_sicredi.silver_cartoes"
    )
)

(
    df_cartoes_quarentena.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.desafio_sicredi.quarentena_cartoes"
    )
)

In [0]:
%sql
SELECT 'clientes' AS entidade,
       COUNT(*) AS total
FROM workspace.desafio_sicredi.silver_clientes

UNION ALL

SELECT 'contas', COUNT(*)
FROM workspace.desafio_sicredi.silver_contas

UNION ALL

SELECT 'cartoes', COUNT(*)
FROM workspace.desafio_sicredi.silver_cartoes;

entidade,total
clientes,2
contas,2
cartoes,3
